# Model 1 — Whole Plate Detector

This notebook trains the first model in the Iranian ALPR pipeline.

Input:
- Full vehicle/scenery images from `Datasets/DS_model_1/train`
- Full vehicle/scenery images from `Datasets/DS_model_1/valid`

Annotation:
- Side-by-side XML files with the same stem as each image
- Pascal/VOC-like object annotations
- XML may not contain `<size>`, `<width>`, or `<height>`
- Image dimensions are always read from the actual image file

Training target:
- YOLO detector
- Class `0`: plate

Strict label rule:
- Only objects whose name is exactly `کل ناحیه پلاک` are used.
- Character boxes are never used for Model 1 training by default.

## Safety Contract

This notebook is intentionally strict:

- No train/valid mixing.
- No test split creation.
- No physical deletion of invalid data.
- No augmentation is added externally.
- XML image size is not trusted.
- Image dimensions are read from image files.
- Only exact `کل ناحیه پلاک` objects are used as plate boxes.
- Multiple plate boxes per image are preserved.
- Samples without valid whole-plate boxes are skipped and logged.
- Character boxes are ignored by default.
- Optional derived enclosing plate box from exactly 8 valid character boxes is disabled by default.
- Bounding boxes are validated against image boundaries.
- Tiny overflow can be clamped and logged.
- Large overflow or malformed boxes are rejected.

In [ ]:
from __future__ import annotations

import os
import re
import cv2
import json
import math
import time
import random
import shutil
import platform
import warnings
import xml.etree.ElementTree as ET
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFile, UnidentifiedImageError

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch

ImageFile.LOAD_TRUNCATED_IMAGES = False

warnings.filterwarnings("default")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Platform:", platform.platform())

In [ ]:
try:
    from ultralytics import YOLO
    import ultralytics
    print("Ultralytics version:", ultralytics.__version__)
except Exception as exc:
    YOLO = None
    print("[ERROR] Could not import ultralytics.")
    print("Install it with:")
    print("    pip install ultralytics")
    raise exc

In [ ]:
@dataclass(frozen=True)
class Model1Config:
    # Paths
    project_root: Path = Path(".").resolve()
    dataset_root: Path = Path("Datasets/DS_model_1")
    train_dir_name: str = "train"
    valid_dir_name: str = "valid"

    yolo_models_dir: Path = Path("yolo models")
    base_yolo_model_name: str = "yolo26s.pt"

    saved_model_dir: Path = Path("Saved_models/model_1")

    # Labels
    plate_label_exact: str = "کل ناحیه پلاک"
    yolo_class_id: int = 0
    yolo_class_name: str = "plate"

    # Character-derived fallback
    derive_plate_from_8_char_boxes: bool = False
    required_character_count_for_derivation: int = 8
    derived_plate_padding_ratio_x: float = 0.04
    derived_plate_padding_ratio_y: float = 0.12

    # Image handling
    valid_image_extensions: Tuple[str, ...] = (
        ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
    )
    ignore_hidden_files: bool = True

    # BBox validation
    tiny_overflow_px: int = 2
    reject_if_box_outside_more_than_tiny: bool = True
    min_box_width_px: int = 4
    min_box_height_px: int = 4
    min_plate_area_ratio: float = 0.00005

    # Generated YOLO dataset
    generated_dataset_dir_name: str = "yolo_dataset"
    generated_images_dir_name: str = "images"
    generated_labels_dir_name: str = "labels"

    # Training
    seed: int = 20260531
    epochs: int = 120
    imgsz: int = 960
    batch: int = 8
    patience: int = 25
    workers: int = 0
    optimizer: str = "AdamW"
    lr0: float = 8e-4
    lrf: float = 0.01
    weight_decay: float = 0.0005
    close_mosaic: int = 0

    # No augmentation
    hsv_h: float = 0.0
    hsv_s: float = 0.0
    hsv_v: float = 0.0
    degrees: float = 0.0
    translate: float = 0.0
    scale: float = 0.0
    shear: float = 0.0
    perspective: float = 0.0
    flipud: float = 0.0
    fliplr: float = 0.0
    mosaic: float = 0.0
    mixup: float = 0.0
    copy_paste: float = 0.0

    # Inference
    inference_conf: float = 0.25
    inference_iou: float = 0.50
    max_det: int = 20
    crop_padding_ratio_x: float = 0.03
    crop_padding_ratio_y: float = 0.08

    # Optional deskew/rectify
    enable_optional_rectification: bool = True
    rectification_min_contour_area_ratio: float = 0.10
    rectification_max_angle_abs_deg: float = 12.0

    # Device
    prefer_mps: bool = True
    allow_cpu_fallback: bool = True
    allow_yolo_cpu_fallback: bool = False

    # Output files
    config_name: str = "config.json"
    data_yaml_name: str = "data.yaml"
    valid_index_name: str = "valid_samples.csv"
    invalid_report_name: str = "invalid_samples.csv"
    bbox_clamp_report_name: str = "bbox_clamp_report.csv"
    conversion_log_name: str = "conversion_log.csv"

    # Visualization
    preview_count: int = 8


CFG = Model1Config()

PROJECT_ROOT = CFG.project_root
DATASET_ROOT = PROJECT_ROOT / CFG.dataset_root
TRAIN_DIR = DATASET_ROOT / CFG.train_dir_name
VALID_DIR = DATASET_ROOT / CFG.valid_dir_name

YOLO_MODELS_DIR = PROJECT_ROOT / CFG.yolo_models_dir
BASE_YOLO_MODEL_PATH = YOLO_MODELS_DIR / CFG.base_yolo_model_name

SAVED_DIR = PROJECT_ROOT / CFG.saved_model_dir
GENERATED_DATASET_DIR = SAVED_DIR / CFG.generated_dataset_dir_name

YOLO_IMAGES_TRAIN_DIR = GENERATED_DATASET_DIR / CFG.generated_images_dir_name / "train"
YOLO_IMAGES_VALID_DIR = GENERATED_DATASET_DIR / CFG.generated_images_dir_name / "valid"

YOLO_LABELS_TRAIN_DIR = GENERATED_DATASET_DIR / CFG.generated_labels_dir_name / "train"
YOLO_LABELS_VALID_DIR = GENERATED_DATASET_DIR / CFG.generated_labels_dir_name / "valid"

DATA_YAML_PATH = SAVED_DIR / CFG.data_yaml_name

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("VALID_DIR:", VALID_DIR)
print("BASE_YOLO_MODEL_PATH:", BASE_YOLO_MODEL_PATH)
print("SAVED_DIR:", SAVED_DIR)

In [ ]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        print(f"[WARN] Could not enable deterministic algorithms: {exc}")


seed_everything(CFG.seed)


def get_torch_device(prefer_mps: bool = True, allow_cpu_fallback: bool = True) -> torch.device:
    if prefer_mps and torch.backends.mps.is_available():
        print("[DEVICE] Using Apple MPS for torch operations.")
        return torch.device("mps")

    if prefer_mps and not torch.backends.mps.is_available():
        msg = "[DEVICE] Apple MPS is not available."
        if allow_cpu_fallback:
            print(msg + " Falling back to CPU.")
            return torch.device("cpu")
        raise RuntimeError(msg + " CPU fallback is disabled.")

    return torch.device("cpu")


DEVICE = get_torch_device(CFG.prefer_mps, CFG.allow_cpu_fallback)


def get_yolo_device_string(cfg: Model1Config) -> str:
    if cfg.prefer_mps and torch.backends.mps.is_available():
        return "mps"

    if cfg.allow_yolo_cpu_fallback:
        print("[YOLO DEVICE WARN] MPS unavailable or disabled. YOLO will use CPU.")
        return "cpu"

    raise RuntimeError(
        "YOLO MPS device is not available and allow_yolo_cpu_fallback=False. "
        "Enable CPU fallback explicitly only if you accept slower training."
    )


YOLO_DEVICE = get_yolo_device_string(CFG)

print("Torch device:", DEVICE)
print("YOLO device:", YOLO_DEVICE)
print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

In [ ]:
def require_dir(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{description} does not exist: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{description} is not a directory: {path}")


def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{description} does not exist: {path}")
    if not path.is_file():
        raise FileNotFoundError(f"{description} is not a file: {path}")


require_dir(DATASET_ROOT, "Model 1 dataset root")
require_dir(TRAIN_DIR, "Model 1 train directory")
require_dir(VALID_DIR, "Model 1 valid directory")
require_dir(YOLO_MODELS_DIR, "YOLO models directory")
require_file(BASE_YOLO_MODEL_PATH, "Base YOLO model")

SAVED_DIR.mkdir(parents=True, exist_ok=True)
(SAVED_DIR / "plots").mkdir(parents=True, exist_ok=True)
(SAVED_DIR / "preview_annotations").mkdir(parents=True, exist_ok=True)
(SAVED_DIR / "inference_examples").mkdir(parents=True, exist_ok=True)

print("[OK] Required paths validated.")

In [ ]:
def print_boxed(title: str) -> None:
    line = "=" * max(80, len(title) + 8)
    print("\n" + line)
    print(title)
    print(line)


def is_supported_image(path: Path, extensions: Tuple[str, ...]) -> bool:
    return path.suffix.lower() in extensions


def pil_get_image_size(path: Path) -> Tuple[int, int]:
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img = img.convert("RGB")
            width, height = img.size

        if width <= 0 or height <= 0:
            raise ValueError(f"Invalid non-positive image size: {(width, height)}")

        return width, height

    except Exception as exc:
        raise RuntimeError(f"Could not read image dimensions from {path}: {exc}") from exc


def load_rgb_image(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return img.convert("RGB")


def safe_json_dump(obj: Any, path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def normalize_label_text(text: Optional[str]) -> str:
    if text is None:
        return ""
    return text.strip()


def ensure_clean_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def clamp_int(v: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, v))

In [ ]:
@dataclass
class RawObjectBox:
    name: str
    xmin: float
    ymin: float
    xmax: float
    ymax: float


@dataclass
class ValidPlateBox:
    name: str
    xmin: int
    ymin: int
    xmax: int
    ymax: int
    center_x: float
    center_y: float
    width: int
    height: int
    source: str
    original_order: int


@dataclass
class ValidSample:
    split: str
    image_path: Path
    xml_path: Path
    width: int
    height: int
    plate_boxes: List[ValidPlateBox]


@dataclass
class InvalidSample:
    split: str
    image_path: Optional[str]
    xml_path: Optional[str]
    reason: str
    detail: str


@dataclass
class ClampEvent:
    split: str
    image_path: str
    xml_path: str
    object_name: str
    before: Tuple[float, float, float, float]
    after: Tuple[int, int, int, int]
    reason: str

In [ ]:
def parse_pascal_voc_xml(xml_path: Path) -> List[RawObjectBox]:
    """
    Parse Pascal/VOC-like XML.

    Important:
    - Do not assume <size>, <width>, or <height>.
    - Image dimensions are read from the image file.
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as exc:
        raise RuntimeError(f"Failed to parse XML {xml_path}: {exc}") from exc

    objects: List[RawObjectBox] = []

    for obj in root.findall(".//object"):
        name = normalize_label_text(obj.findtext("name"))
        bndbox = obj.find("bndbox")

        if bndbox is None:
            raise ValueError(f"Object without bndbox in {xml_path}")

        def read_coord(tag: str) -> float:
            value = bndbox.findtext(tag)
            if value is None:
                raise ValueError(f"Missing {tag} in bndbox in {xml_path}")
            value = value.strip()
            if value == "":
                raise ValueError(f"Empty {tag} in bndbox in {xml_path}")
            return float(value)

        objects.append(
            RawObjectBox(
                name=name,
                xmin=read_coord("xmin"),
                ymin=read_coord("ymin"),
                xmax=read_coord("xmax"),
                ymax=read_coord("ymax"),
            )
        )

    return objects

In [ ]:
def validate_and_maybe_clamp_box(
    raw: RawObjectBox,
    image_width: int,
    image_height: int,
    cfg: Model1Config,
) -> Tuple[Optional[Tuple[int, int, int, int]], Optional[str], Optional[str]]:
    values = [raw.xmin, raw.ymin, raw.xmax, raw.ymax]

    if any(not np.isfinite(v) for v in values):
        return None, "Non-finite bbox coordinate", None

    xmin, ymin, xmax, ymax = values

    if xmax <= xmin or ymax <= ymin:
        return None, f"Non-positive bbox area before clamp: {(xmin, ymin, xmax, ymax)}", None

    overflow_left = max(0.0, -xmin)
    overflow_top = max(0.0, -ymin)
    overflow_right = max(0.0, xmax - image_width)
    overflow_bottom = max(0.0, ymax - image_height)
    max_overflow = max(overflow_left, overflow_top, overflow_right, overflow_bottom)

    clamp_reason = None

    if max_overflow > 0:
        if max_overflow <= cfg.tiny_overflow_px:
            clamp_reason = f"Tiny overflow clamped: max_overflow={max_overflow:.3f}px"
        else:
            if cfg.reject_if_box_outside_more_than_tiny:
                return None, (
                    f"BBox outside image by more than tiny threshold. "
                    f"bbox={(xmin, ymin, xmax, ymax)}, "
                    f"image_size={(image_width, image_height)}, "
                    f"max_overflow={max_overflow:.3f}px"
                ), None

    xmin_i = int(round(max(0, min(image_width - 1, xmin))))
    ymin_i = int(round(max(0, min(image_height - 1, ymin))))
    xmax_i = int(round(max(0, min(image_width, xmax))))
    ymax_i = int(round(max(0, min(image_height, ymax))))

    if xmax_i <= xmin_i or ymax_i <= ymin_i:
        return None, f"Non-positive bbox area after clamp: {(xmin_i, ymin_i, xmax_i, ymax_i)}", clamp_reason

    bw = xmax_i - xmin_i
    bh = ymax_i - ymin_i

    if bw < cfg.min_box_width_px or bh < cfg.min_box_height_px:
        return None, f"BBox too small: width={bw}, height={bh}", clamp_reason

    area_ratio = (bw * bh) / float(image_width * image_height)

    if area_ratio < cfg.min_plate_area_ratio:
        return None, f"Plate bbox area ratio too small: {area_ratio:.8f}", clamp_reason

    return (xmin_i, ymin_i, xmax_i, ymax_i), None, clamp_reason

In [ ]:
def derive_plate_box_from_character_boxes(
    raw_objects: List[RawObjectBox],
    image_width: int,
    image_height: int,
    cfg: Model1Config,
) -> Optional[RawObjectBox]:
    """
    Disabled by default.

    If enabled, derive one enclosing plate box only when exactly 8 valid non-plate boxes exist.
    This is a fallback for samples that have character boxes but no explicit whole-plate box.
    """
    if not cfg.derive_plate_from_8_char_boxes:
        return None

    char_candidates = [
        obj for obj in raw_objects
        if normalize_label_text(obj.name) != cfg.plate_label_exact
    ]

    if len(char_candidates) != cfg.required_character_count_for_derivation:
        return None

    valid_coords = []

    for obj in char_candidates:
        bbox, rejection, _ = validate_and_maybe_clamp_box(
            raw=obj,
            image_width=image_width,
            image_height=image_height,
            cfg=cfg,
        )
        if bbox is None:
            return None
        valid_coords.append(bbox)

    xmin = min(b[0] for b in valid_coords)
    ymin = min(b[1] for b in valid_coords)
    xmax = max(b[2] for b in valid_coords)
    ymax = max(b[3] for b in valid_coords)

    bw = xmax - xmin
    bh = ymax - ymin

    pad_x = int(round(bw * cfg.derived_plate_padding_ratio_x))
    pad_y = int(round(bh * cfg.derived_plate_padding_ratio_y))

    xmin = max(0, xmin - pad_x)
    ymin = max(0, ymin - pad_y)
    xmax = min(image_width, xmax + pad_x)
    ymax = min(image_height, ymax + pad_y)

    if xmax <= xmin or ymax <= ymin:
        return None

    return RawObjectBox(
        name=cfg.plate_label_exact,
        xmin=float(xmin),
        ymin=float(ymin),
        xmax=float(xmax),
        ymax=float(ymax),
    )

In [ ]:
def discover_image_xml_pairs(
    split_dir: Path,
    split_name: str,
    cfg: Model1Config,
) -> Tuple[List[Tuple[Path, Path]], List[InvalidSample]]:
    invalids: List[InvalidSample] = []
    pairs: List[Tuple[Path, Path]] = []

    image_paths: List[Path] = []

    for path in sorted(split_dir.iterdir(), key=lambda p: p.name):
        if cfg.ignore_hidden_files and path.name.startswith("."):
            continue

        if path.is_dir():
            invalids.append(
                InvalidSample(
                    split=split_name,
                    image_path=None,
                    xml_path=None,
                    reason="Unexpected subdirectory",
                    detail=str(path),
                )
            )
            continue

        if is_supported_image(path, cfg.valid_image_extensions):
            image_paths.append(path)

    for image_path in image_paths:
        xml_path = image_path.with_suffix(".xml")

        if not xml_path.exists():
            invalids.append(
                InvalidSample(
                    split=split_name,
                    image_path=str(image_path),
                    xml_path=str(xml_path),
                    reason="Missing side-by-side XML",
                    detail="Expected XML with same stem as image",
                )
            )
            continue

        pairs.append((image_path, xml_path))

    return pairs, invalids


train_pairs, train_pair_invalids = discover_image_xml_pairs(TRAIN_DIR, "train", CFG)
valid_pairs, valid_pair_invalids = discover_image_xml_pairs(VALID_DIR, "valid", CFG)

print_boxed("Pair discovery summary")
print(f"Train image/XML pairs found: {len(train_pairs):,}")
print(f"Valid image/XML pairs found: {len(valid_pairs):,}")
print(f"Initial invalid pair issues: {len(train_pair_invalids) + len(valid_pair_invalids):,}")

In [ ]:
def build_valid_samples_for_split(
    pairs: List[Tuple[Path, Path]],
    split_name: str,
    cfg: Model1Config,
) -> Tuple[List[ValidSample], List[InvalidSample], List[ClampEvent]]:
    valid_samples: List[ValidSample] = []
    invalids: List[InvalidSample] = []
    clamp_events: List[ClampEvent] = []

    for image_path, xml_path in tqdm(pairs, desc=f"Validating {split_name}"):
        try:
            width, height = pil_get_image_size(image_path)
        except Exception as exc:
            invalids.append(
                InvalidSample(
                    split=split_name,
                    image_path=str(image_path),
                    xml_path=str(xml_path),
                    reason="Image unreadable or invalid",
                    detail=repr(exc),
                )
            )
            continue

        try:
            raw_objects = parse_pascal_voc_xml(xml_path)
        except Exception as exc:
            invalids.append(
                InvalidSample(
                    split=split_name,
                    image_path=str(image_path),
                    xml_path=str(xml_path),
                    reason="XML parse failure",
                    detail=repr(exc),
                )
            )
            continue

        plate_raw_objects = [
            obj for obj in raw_objects
            if normalize_label_text(obj.name) == cfg.plate_label_exact
        ]

        derived_used = False

        if len(plate_raw_objects) == 0 and cfg.derive_plate_from_8_char_boxes:
            derived = derive_plate_box_from_character_boxes(
                raw_objects=raw_objects,
                image_width=width,
                image_height=height,
                cfg=cfg,
            )
            if derived is not None:
                plate_raw_objects = [derived]
                derived_used = True

        if len(plate_raw_objects) == 0:
            invalids.append(
                InvalidSample(
                    split=split_name,
                    image_path=str(image_path),
                    xml_path=str(xml_path),
                    reason="No valid whole-plate object",
                    detail=(
                        f"No object with exact name '{cfg.plate_label_exact}'. "
                        f"Derivation enabled={cfg.derive_plate_from_8_char_boxes}"
                    ),
                )
            )
            continue

        valid_plate_boxes: List[ValidPlateBox] = []
        rejection_reasons: List[str] = []

        for obj_idx, raw in enumerate(plate_raw_objects):
            bbox, rejection_reason, clamp_reason = validate_and_maybe_clamp_box(
                raw=raw,
                image_width=width,
                image_height=height,
                cfg=cfg,
            )

            if clamp_reason is not None and bbox is not None:
                clamp_events.append(
                    ClampEvent(
                        split=split_name,
                        image_path=str(image_path),
                        xml_path=str(xml_path),
                        object_name=raw.name,
                        before=(raw.xmin, raw.ymin, raw.xmax, raw.ymax),
                        after=bbox,
                        reason=clamp_reason,
                    )
                )

            if bbox is None:
                rejection_reasons.append(
                    f"plate_object_index={obj_idx}, reason={rejection_reason}"
                )
                continue

            xmin, ymin, xmax, ymax = bbox

            valid_plate_boxes.append(
                ValidPlateBox(
                    name=raw.name,
                    xmin=xmin,
                    ymin=ymin,
                    xmax=xmax,
                    ymax=ymax,
                    center_x=(xmin + xmax) / 2.0,
                    center_y=(ymin + ymax) / 2.0,
                    width=xmax - xmin,
                    height=ymax - ymin,
                    source="derived_from_characters" if derived_used else "explicit_plate_label",
                    original_order=obj_idx,
                )
            )

        if len(valid_plate_boxes) == 0:
            invalids.append(
                InvalidSample(
                    split=split_name,
                    image_path=str(image_path),
                    xml_path=str(xml_path),
                    reason="All whole-plate boxes invalid",
                    detail=" | ".join(rejection_reasons),
                )
            )
            continue

        valid_samples.append(
            ValidSample(
                split=split_name,
                image_path=image_path,
                xml_path=xml_path,
                width=width,
                height=height,
                plate_boxes=valid_plate_boxes,
            )
        )

    return valid_samples, invalids, clamp_events


train_valid_samples, train_invalid_samples, train_clamps = build_valid_samples_for_split(
    train_pairs, "train", CFG
)

valid_valid_samples, valid_invalid_samples, valid_clamps = build_valid_samples_for_split(
    valid_pairs, "valid", CFG
)

all_valid_samples = train_valid_samples + valid_valid_samples

all_invalid_samples = (
    train_pair_invalids
    + valid_pair_invalids
    + train_invalid_samples
    + valid_invalid_samples
)

all_clamp_events = train_clamps + valid_clamps

print_boxed("Strict validation summary")
print(f"Valid train samples: {len(train_valid_samples):,}")
print(f"Valid valid samples: {len(valid_valid_samples):,}")
print(f"Invalid samples/issues: {len(all_invalid_samples):,}")
print(f"Tiny clamp events: {len(all_clamp_events):,}")

if len(train_valid_samples) == 0:
    raise RuntimeError("No valid train samples after strict filtering.")

if len(valid_valid_samples) == 0:
    raise RuntimeError("No valid valid samples after strict filtering.")

In [ ]:
def valid_samples_to_rows(samples: List[ValidSample]) -> List[Dict[str, Any]]:
    rows = []

    for sample in samples:
        row = {
            "split": sample.split,
            "image_path": str(sample.image_path),
            "xml_path": str(sample.xml_path),
            "width": sample.width,
            "height": sample.height,
            "num_plate_boxes": len(sample.plate_boxes),
        }

        for i, box in enumerate(sample.plate_boxes, start=1):
            prefix = f"P{i}"
            row[f"{prefix}_xmin"] = box.xmin
            row[f"{prefix}_ymin"] = box.ymin
            row[f"{prefix}_xmax"] = box.xmax
            row[f"{prefix}_ymax"] = box.ymax
            row[f"{prefix}_width"] = box.width
            row[f"{prefix}_height"] = box.height
            row[f"{prefix}_source"] = box.source

        rows.append(row)

    return rows


valid_index_df = pd.DataFrame(valid_samples_to_rows(all_valid_samples))
invalid_df = pd.DataFrame([asdict(x) for x in all_invalid_samples])
clamp_df = pd.DataFrame([asdict(x) for x in all_clamp_events])

valid_index_path = SAVED_DIR / CFG.valid_index_name
invalid_report_path = SAVED_DIR / CFG.invalid_report_name
clamp_report_path = SAVED_DIR / CFG.bbox_clamp_report_name

valid_index_df.to_csv(valid_index_path, index=False, encoding="utf-8-sig")
invalid_df.to_csv(invalid_report_path, index=False, encoding="utf-8-sig")
clamp_df.to_csv(clamp_report_path, index=False, encoding="utf-8-sig")

print("Valid index saved to:", valid_index_path)
print("Invalid report saved to:", invalid_report_path)
print("Clamp report saved to:", clamp_report_path)

display(valid_index_df.head())
display(invalid_df.head())

In [ ]:
def draw_plate_boxes_on_image(
    image: Image.Image,
    boxes: List[ValidPlateBox],
) -> Image.Image:
    img = image.copy()
    draw = ImageDraw.Draw(img)

    for i, box in enumerate(boxes, start=1):
        color = "lime" if box.source == "explicit_plate_label" else "orange"

        draw.rectangle([box.xmin, box.ymin, box.xmax, box.ymax], outline=color, width=3)
        draw.text(
            (box.xmin, max(0, box.ymin - 16)),
            f"P{i} {box.source}",
            fill=color,
        )

    return img


def visualize_valid_samples(
    samples: List[ValidSample],
    count: int,
    save_dir: Path,
    seed: int,
) -> None:
    rng = random.Random(seed)
    chosen = rng.sample(samples, k=min(count, len(samples)))

    cols = 2
    rows = math.ceil(len(chosen) / cols)

    plt.figure(figsize=(cols * 9, rows * 6))

    for idx, sample in enumerate(chosen):
        img = load_rgb_image(sample.image_path)
        vis = draw_plate_boxes_on_image(img, sample.plate_boxes)

        out_path = save_dir / f"{sample.split}_{idx:03d}_{sample.image_path.stem}.png"
        vis.save(out_path)

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(vis)
        plt.title(f"{sample.split} | {sample.image_path.name}")
        plt.axis("off")

    plt.tight_layout()
    plt.savefig(SAVED_DIR / "plots" / "model1_annotation_preview.png", dpi=160)
    plt.show()


visualize_valid_samples(
    samples=all_valid_samples,
    count=CFG.preview_count,
    save_dir=SAVED_DIR / "preview_annotations",
    seed=CFG.seed,
)

In [ ]:
def voc_plate_box_to_yolo_line(
    box: ValidPlateBox,
    image_width: int,
    image_height: int,
    class_id: int,
) -> str:
    x_center = ((box.xmin + box.xmax) / 2.0) / image_width
    y_center = ((box.ymin + box.ymax) / 2.0) / image_height
    width = (box.xmax - box.xmin) / image_width
    height = (box.ymax - box.ymin) / image_height

    vals = [x_center, y_center, width, height]

    if any(v <= 0 or v > 1 for v in vals):
        raise ValueError(
            f"Invalid YOLO normalized plate bbox: {vals}, "
            f"box={box}, image_size={(image_width, image_height)}"
        )

    return f"{class_id} {x_center:.8f} {y_center:.8f} {width:.8f} {height:.8f}"

In [ ]:
def prepare_yolo_dataset_dirs() -> None:
    ensure_clean_dir(GENERATED_DATASET_DIR)

    YOLO_IMAGES_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    YOLO_IMAGES_VALID_DIR.mkdir(parents=True, exist_ok=True)

    YOLO_LABELS_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    YOLO_LABELS_VALID_DIR.mkdir(parents=True, exist_ok=True)


def copy_image_and_write_plate_labels(
    sample: ValidSample,
    cfg: Model1Config,
) -> Dict[str, Any]:
    if sample.split == "train":
        image_out_dir = YOLO_IMAGES_TRAIN_DIR
        label_out_dir = YOLO_LABELS_TRAIN_DIR
    elif sample.split == "valid":
        image_out_dir = YOLO_IMAGES_VALID_DIR
        label_out_dir = YOLO_LABELS_VALID_DIR
    else:
        raise ValueError(f"Unexpected split: {sample.split}")

    out_image_path = image_out_dir / sample.image_path.name
    out_label_path = label_out_dir / f"{sample.image_path.stem}.txt"

    shutil.copy2(sample.image_path, out_image_path)

    lines = [
        voc_plate_box_to_yolo_line(
            box=box,
            image_width=sample.width,
            image_height=sample.height,
            class_id=cfg.yolo_class_id,
        )
        for box in sample.plate_boxes
    ]

    if len(lines) == 0:
        raise RuntimeError(f"No plate labels for sample: {sample.image_path}")

    out_label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    return {
        "split": sample.split,
        "source_image": str(sample.image_path),
        "source_xml": str(sample.xml_path),
        "yolo_image": str(out_image_path),
        "yolo_label": str(out_label_path),
        "num_plate_boxes": len(lines),
        "width": sample.width,
        "height": sample.height,
    }


prepare_yolo_dataset_dirs()

conversion_rows = []

for sample in tqdm(all_valid_samples, desc="Generating YOLO dataset"):
    conversion_rows.append(copy_image_and_write_plate_labels(sample, CFG))

conversion_df = pd.DataFrame(conversion_rows)
conversion_log_path = SAVED_DIR / CFG.conversion_log_name
conversion_df.to_csv(conversion_log_path, index=False, encoding="utf-8-sig")

print("[OK] YOLO dataset generated.")
print("Conversion log:", conversion_log_path)
print("Train images:", len(list(YOLO_IMAGES_TRAIN_DIR.glob("*"))))
print("Train labels:", len(list(YOLO_LABELS_TRAIN_DIR.glob("*.txt"))))
print("Valid images:", len(list(YOLO_IMAGES_VALID_DIR.glob("*"))))
print("Valid labels:", len(list(YOLO_LABELS_VALID_DIR.glob("*.txt"))))

display(conversion_df.head())

In [ ]:
def write_yolo_data_yaml(path: Path, cfg: Model1Config) -> None:
    yaml_text = f"""
path: {GENERATED_DATASET_DIR.as_posix()}
train: {YOLO_IMAGES_TRAIN_DIR.as_posix()}
val: {YOLO_IMAGES_VALID_DIR.as_posix()}

nc: 1
names:
  0: {cfg.yolo_class_name}
"""
    path.write_text(yaml_text, encoding="utf-8")


write_yolo_data_yaml(DATA_YAML_PATH, CFG)

print(DATA_YAML_PATH.read_text(encoding="utf-8"))

In [ ]:
def serialize_config(cfg: Model1Config) -> Dict[str, Any]:
    data = asdict(cfg)

    for k, v in list(data.items()):
        if isinstance(v, Path):
            data[k] = str(v)
        elif isinstance(v, tuple):
            data[k] = list(v)

    data["resolved_paths"] = {
        "project_root": str(PROJECT_ROOT),
        "dataset_root": str(DATASET_ROOT),
        "train_dir": str(TRAIN_DIR),
        "valid_dir": str(VALID_DIR),
        "base_yolo_model_path": str(BASE_YOLO_MODEL_PATH),
        "saved_dir": str(SAVED_DIR),
        "generated_dataset_dir": str(GENERATED_DATASET_DIR),
        "data_yaml": str(DATA_YAML_PATH),
    }

    return data


safe_json_dump(serialize_config(CFG), SAVED_DIR / CFG.config_name)

print("[OK] Config saved:", SAVED_DIR / CFG.config_name)

In [ ]:
def read_yolo_label_file(label_path: Path) -> List[List[float]]:
    rows = []

    text = label_path.read_text(encoding="utf-8").strip()

    if not text:
        raise ValueError(f"Empty YOLO label file: {label_path}")

    for line in text.splitlines():
        parts = line.strip().split()

        if len(parts) != 5:
            raise ValueError(f"Invalid YOLO label line in {label_path}: {line}")

        class_id = int(parts[0])
        vals = [float(x) for x in parts[1:]]

        if class_id != CFG.yolo_class_id:
            raise ValueError(f"Unexpected class id in {label_path}: {class_id}")

        if any(v <= 0 or v > 1 for v in vals):
            raise ValueError(f"Invalid normalized bbox values in {label_path}: {vals}")

        rows.append([class_id] + vals)

    return rows


def sanity_check_yolo_labels(label_dir: Path) -> None:
    label_files = sorted(label_dir.glob("*.txt"))

    if len(label_files) == 0:
        raise RuntimeError(f"No YOLO label files found in {label_dir}")

    bad_files = []

    for label_path in label_files:
        try:
            rows = read_yolo_label_file(label_path)

            if len(rows) < 1:
                bad_files.append((str(label_path), "No boxes"))

        except Exception as exc:
            bad_files.append((str(label_path), repr(exc)))

    if bad_files:
        for item in bad_files[:20]:
            print(item)
        raise RuntimeError(f"YOLO label sanity check failed for {len(bad_files)} files.")

    print(f"[OK] All labels in {label_dir} passed sanity check.")


sanity_check_yolo_labels(YOLO_LABELS_TRAIN_DIR)
sanity_check_yolo_labels(YOLO_LABELS_VALID_DIR)

In [ ]:
def yolo_to_voc(
    x_center: float,
    y_center: float,
    width: float,
    height: float,
    img_w: int,
    img_h: int,
) -> Tuple[int, int, int, int]:
    cx = x_center * img_w
    cy = y_center * img_h
    bw = width * img_w
    bh = height * img_h

    xmin = int(round(cx - bw / 2))
    ymin = int(round(cy - bh / 2))
    xmax = int(round(cx + bw / 2))
    ymax = int(round(cy + bh / 2))

    return xmin, ymin, xmax, ymax


def visualize_yolo_plate_sample(image_path: Path, label_path: Path) -> Image.Image:
    img = load_rgb_image(image_path)
    img_w, img_h = img.size

    draw = ImageDraw.Draw(img)

    rows = read_yolo_label_file(label_path)

    for i, row in enumerate(rows, start=1):
        _, x_center, y_center, bw, bh = row
        xmin, ymin, xmax, ymax = yolo_to_voc(x_center, y_center, bw, bh, img_w, img_h)

        draw.rectangle([xmin, ymin, xmax, ymax], outline="yellow", width=3)
        draw.text((xmin, max(0, ymin - 16)), f"P{i}", fill="yellow")

    return img


sample_label_files = sorted(YOLO_LABELS_TRAIN_DIR.glob("*.txt"))[:CFG.preview_count]

cols = 2
rows = math.ceil(len(sample_label_files) / cols)

plt.figure(figsize=(cols * 9, rows * 6))

for idx, label_path in enumerate(sample_label_files):
    image_candidates = list(YOLO_IMAGES_TRAIN_DIR.glob(label_path.stem + ".*"))
    if not image_candidates:
        continue

    image_path = image_candidates[0]
    vis = visualize_yolo_plate_sample(image_path, label_path)

    plt.subplot(rows, cols, idx + 1)
    plt.imshow(vis)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.savefig(SAVED_DIR / "plots" / "model1_yolo_conversion_preview.png", dpi=160)
plt.show()

## YOLO Training Note

The notebook requests MPS if available.

If Ultralytics/PyTorch does not support your MPS setup properly, this notebook will not silently switch to CPU.

To use CPU intentionally, set:

allow_yolo_cpu_fallback=True

No CUDA-only `.cuda()` code is used.

In [ ]:
allow_yolo_cpu_fallback=True

print("Loading base YOLO model from:", BASE_YOLO_MODEL_PATH)

yolo_model = YOLO(str(BASE_YOLO_MODEL_PATH))

print("[OK] YOLO model loaded.")

In [ ]:
print_boxed("Starting YOLO training for Model 1")
print("Data YAML:", DATA_YAML_PATH)
print("Device:", YOLO_DEVICE)
print("Base model:", BASE_YOLO_MODEL_PATH)
print("Save project:", SAVED_DIR)

try:
    train_results = yolo_model.train(
        data=str(DATA_YAML_PATH),
        epochs=CFG.epochs,
        imgsz=CFG.imgsz,
        batch=CFG.batch,
        patience=CFG.patience,
        workers=CFG.workers,
        device=YOLO_DEVICE,
        project=str(SAVED_DIR),
        name="train_run",
        exist_ok=True,
        seed=CFG.seed,
        optimizer=CFG.optimizer,
        lr0=CFG.lr0,
        lrf=CFG.lrf,
        weight_decay=CFG.weight_decay,
        close_mosaic=CFG.close_mosaic,

        hsv_h=CFG.hsv_h,
        hsv_s=CFG.hsv_s,
        hsv_v=CFG.hsv_v,
        degrees=CFG.degrees,
        translate=CFG.translate,
        scale=CFG.scale,
        shear=CFG.shear,
        perspective=CFG.perspective,
        flipud=CFG.flipud,
        fliplr=CFG.fliplr,
        mosaic=CFG.mosaic,
        mixup=CFG.mixup,
        copy_paste=CFG.copy_paste,

        verbose=True,
    )

    print("[OK] YOLO training finished.")

except Exception as exc:
    print("[ERROR] YOLO training failed.")
    print("Device requested:", YOLO_DEVICE)
    print("Error type:", type(exc).__name__)
    print("Error:", exc)

    if YOLO_DEVICE == "mps":
        print(
            "\n[IMPORTANT] MPS was requested. "
            "If this Ultralytics/PyTorch combination cannot train on MPS, "
            "do NOT silently switch to CPU. "
            "Set allow_yolo_cpu_fallback=True explicitly if you accept CPU training."
        )

    raise

In [ ]:
TRAIN_RUN_DIR = SAVED_DIR / "train_run"
WEIGHTS_DIR = TRAIN_RUN_DIR / "weights"

MODEL1_BEST_PT = WEIGHTS_DIR / "best.pt"
MODEL1_LAST_PT = WEIGHTS_DIR / "last.pt"

if not MODEL1_BEST_PT.exists():
    raise FileNotFoundError(f"YOLO best.pt not found: {MODEL1_BEST_PT}")

if not MODEL1_LAST_PT.exists():
    print("[WARN] YOLO last.pt not found:", MODEL1_LAST_PT)

print("Best weights:", MODEL1_BEST_PT)
print("Last weights:", MODEL1_LAST_PT)

In [ ]:
best_yolo_model = YOLO(str(MODEL1_BEST_PT))

print_boxed("Running YOLO validation")

val_results = best_yolo_model.val(
    data=str(DATA_YAML_PATH),
    imgsz=CFG.imgsz,
    batch=CFG.batch,
    device=YOLO_DEVICE,
    workers=CFG.workers,
    conf=CFG.inference_conf,
    iou=CFG.inference_iou,
    verbose=True,
)

print("[OK] Validation finished.")

metrics_summary = {}

try:
    metrics_summary["box_map"] = float(val_results.box.map)
    metrics_summary["box_map50"] = float(val_results.box.map50)
    metrics_summary["box_map75"] = float(val_results.box.map75)
except Exception as exc:
    metrics_summary["warning"] = f"Could not extract mAP metrics: {repr(exc)}"

safe_json_dump(metrics_summary, SAVED_DIR / "validation_metrics.json")

print(metrics_summary)

In [ ]:
@dataclass
class DetectedPlate:
    plate_id: int
    xmin: int
    ymin: int
    xmax: int
    ymax: int
    center_x: float
    center_y: float
    width: int
    height: int
    confidence: float

In [ ]:
def crop_plate_with_padding(
    image: Image.Image,
    det: DetectedPlate,
    cfg: Model1Config,
) -> Image.Image:
    w, h = image.size

    pad_x = int(round(det.width * cfg.crop_padding_ratio_x))
    pad_y = int(round(det.height * cfg.crop_padding_ratio_y))

    xmin = clamp_int(det.xmin - pad_x, 0, w - 1)
    ymin = clamp_int(det.ymin - pad_y, 0, h - 1)
    xmax = clamp_int(det.xmax + pad_x, 1, w)
    ymax = clamp_int(det.ymax + pad_y, 1, h)

    if xmax <= xmin or ymax <= ymin:
        raise ValueError(f"Invalid plate crop coordinates: {(xmin, ymin, xmax, ymax)}")

    return image.crop((xmin, ymin, xmax, ymax))


def pil_to_cv_rgb(image: Image.Image) -> np.ndarray:
    return np.array(image.convert("RGB"))


def cv_rgb_to_pil(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(arr.astype(np.uint8), mode="RGB")


def try_rectify_plate_classical_cv(
    plate_img: Image.Image,
    cfg: Model1Config,
) -> Tuple[Image.Image, Dict[str, Any]]:
    """
    Conservative deskew/rectification.

    It only applies rotation correction when:
    - A plausible dominant contour/angle is found.
    - Angle magnitude is within configured bound.
    - Otherwise returns original image.

    This avoids being too clever. Clever bugs wear tiny sunglasses.
    """
    if not cfg.enable_optional_rectification:
        return plate_img, {"applied": False, "reason": "Disabled by config"}

    rgb = pil_to_cv_rgb(plate_img)
    h, w = rgb.shape[:2]

    if w < 20 or h < 10:
        return plate_img, {"applied": False, "reason": "Crop too small"}

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    edges = cv2.Canny(gray, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return plate_img, {"applied": False, "reason": "No contours"}

    image_area = float(w * h)

    contour = max(contours, key=cv2.contourArea)
    contour_area = cv2.contourArea(contour)

    if contour_area / image_area < cfg.rectification_min_contour_area_ratio:
        return plate_img, {
            "applied": False,
            "reason": "Largest contour too small",
            "contour_area_ratio": contour_area / image_area,
        }

    rect = cv2.minAreaRect(contour)
    angle = rect[-1]

    if angle < -45:
        angle = 90 + angle

    if abs(angle) > cfg.rectification_max_angle_abs_deg:
        return plate_img, {
            "applied": False,
            "reason": "Angle too large for safe correction",
            "angle": float(angle),
        }

    if abs(angle) < 1.0:
        return plate_img, {
            "applied": False,
            "reason": "Angle too small",
            "angle": float(angle),
        }

    center = (w / 2.0, h / 2.0)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)

    rotated = cv2.warpAffine(
        rgb,
        rot_mat,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )

    return cv_rgb_to_pil(rotated), {
        "applied": True,
        "angle": float(angle),
        "contour_area_ratio": contour_area / image_area,
    }

In [ ]:
def detections_from_yolo_result(
    result: Any,
    image_width: int,
    image_height: int,
) -> List[DetectedPlate]:
    boxes = result.boxes

    detections: List[DetectedPlate] = []

    if boxes is None or len(boxes) == 0:
        return detections

    xyxy = boxes.xyxy.detach().cpu().numpy()
    confs = boxes.conf.detach().cpu().numpy()

    for i in range(len(xyxy)):
        x1, y1, x2, y2 = xyxy[i]
        conf = float(confs[i])

        xmin = clamp_int(int(round(x1)), 0, image_width - 1)
        ymin = clamp_int(int(round(y1)), 0, image_height - 1)
        xmax = clamp_int(int(round(x2)), 1, image_width)
        ymax = clamp_int(int(round(y2)), 1, image_height)

        if xmax <= xmin or ymax <= ymin:
            continue

        detections.append(
            DetectedPlate(
                plate_id=-1,
                xmin=xmin,
                ymin=ymin,
                xmax=xmax,
                ymax=ymax,
                center_x=(xmin + xmax) / 2.0,
                center_y=(ymin + ymax) / 2.0,
                width=xmax - xmin,
                height=ymax - ymin,
                confidence=conf,
            )
        )

    detections = sorted(detections, key=lambda d: (d.center_y, d.center_x))

    for idx, det in enumerate(detections, start=1):
        det.plate_id = idx

    return detections


def run_model1_on_image(
    image_path: Path,
    model: YOLO,
    cfg: Model1Config,
    device: str,
) -> Dict[str, Any]:
    if not image_path.exists():
        raise FileNotFoundError(f"Input image does not exist: {image_path}")

    full_img = load_rgb_image(image_path)
    image_width, image_height = full_img.size

    results = model.predict(
        source=str(image_path),
        imgsz=cfg.imgsz,
        conf=cfg.inference_conf,
        iou=cfg.inference_iou,
        max_det=cfg.max_det,
        device=device,
        verbose=False,
    )

    if len(results) != 1:
        raise RuntimeError(f"Expected one result for one image, got {len(results)}")

    detections = detections_from_yolo_result(
        result=results[0],
        image_width=image_width,
        image_height=image_height,
    )

    crops = []
    rectified_crops = []
    rectification_info = []

    for det in detections:
        crop = crop_plate_with_padding(full_img, det, cfg)
        rectified, info = try_rectify_plate_classical_cv(crop, cfg)

        crops.append(crop)
        rectified_crops.append(rectified)
        rectification_info.append(info)

    accepted = len(detections) > 0

    return {
        "accepted": accepted,
        "rejection_reason": None if accepted else "Model 1 found no plates.",
        "image": full_img,
        "detections": detections,
        "crops": crops,
        "rectified_crops": rectified_crops,
        "rectification_info": rectification_info,
        "raw_result": results[0],
    }

In [ ]:
def visualize_model1_inference(result: Dict[str, Any], title: str = "Model 1 Inference") -> None:
    full_img: Image.Image = result["image"]
    detections: List[DetectedPlate] = result["detections"]

    vis = full_img.copy()
    draw = ImageDraw.Draw(vis)

    for det in detections:
        draw.rectangle([det.xmin, det.ymin, det.xmax, det.ymax], outline="lime", width=4)
        draw.text(
            (det.xmin, max(0, det.ymin - 18)),
            f"P{det.plate_id} {det.confidence:.2f}",
            fill="lime",
        )

    plt.figure(figsize=(14, 8))
    plt.imshow(vis)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    if len(detections) == 0:
        print("[REJECTED]", result["rejection_reason"])
        return

    crops = result["crops"]
    rectified_crops = result["rectified_crops"]
    rectification_info = result["rectification_info"]

    n = len(crops)
    plt.figure(figsize=(min(5 * n, 20), 6))

    for i, crop in enumerate(crops, start=1):
        plt.subplot(2, n, i)
        plt.imshow(crop)
        plt.title(f"Raw P{i}")
        plt.axis("off")

    for i, crop in enumerate(rectified_crops, start=1):
        plt.subplot(2, n, n + i)
        plt.imshow(crop)
        info = rectification_info[i - 1]
        applied = info.get("applied", False)
        plt.title(f"Rectified P{i} | applied={applied}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


def print_plate_detection_metadata(detections: List[DetectedPlate]) -> None:
    rows = []

    for det in detections:
        rows.append({
            "plate_id": f"P{det.plate_id}",
            "xmin": det.xmin,
            "ymin": det.ymin,
            "xmax": det.xmax,
            "ymax": det.ymax,
            "center_x": det.center_x,
            "center_y": det.center_y,
            "width": det.width,
            "height": det.height,
            "confidence": det.confidence,
        })

    df = pd.DataFrame(rows)
    display(df)

In [ ]:
sample = random.Random(CFG.seed).choice(valid_valid_samples)

print("Sample full image:", sample.image_path)

result = run_model1_on_image(
    image_path=sample.image_path,
    model=best_yolo_model,
    cfg=CFG,
    device=YOLO_DEVICE,
)

visualize_model1_inference(result, title=f"Model 1 Test: {sample.image_path.name}")
print_plate_detection_metadata(result["detections"])

print("Rectification info:")
display(pd.DataFrame(result["rectification_info"]))

In [ ]:
def evaluate_plate_detection_presence(
    samples: List[ValidSample],
    model: YOLO,
    cfg: Model1Config,
    device: str,
    limit: Optional[int] = None,
) -> pd.DataFrame:
    eval_samples = samples if limit is None else samples[:limit]
    rows = []

    for sample in tqdm(eval_samples, desc="Evaluating plate detection presence"):
        try:
            result = run_model1_on_image(
                image_path=sample.image_path,
                model=model,
                cfg=cfg,
                device=device,
            )

            detections = result["detections"]

            mean_conf = float(np.mean([d.confidence for d in detections])) if detections else 0.0

            rows.append({
                "image_path": str(sample.image_path),
                "gt_plate_count": len(sample.plate_boxes),
                "detected_plate_count": len(detections),
                "accepted": len(detections) > 0,
                "mean_confidence": mean_conf,
                "rejection_reason": result["rejection_reason"],
            })

        except Exception as exc:
            rows.append({
                "image_path": str(sample.image_path),
                "gt_plate_count": len(sample.plate_boxes),
                "detected_plate_count": -1,
                "accepted": False,
                "mean_confidence": 0.0,
                "rejection_reason": repr(exc),
            })

    return pd.DataFrame(rows)


presence_df = evaluate_plate_detection_presence(
    samples=valid_valid_samples,
    model=best_yolo_model,
    cfg=CFG,
    device=YOLO_DEVICE,
    limit=None,
)

presence_path = SAVED_DIR / "validation_detection_presence.csv"
presence_df.to_csv(presence_path, index=False, encoding="utf-8-sig")

display(presence_df.head())

presence_rate = presence_df["accepted"].mean()
print(f"Validation detection presence rate: {presence_rate:.4f}")
print("Saved to:", presence_path)

In [ ]:
def save_model1_inference_visual(
    result: Dict[str, Any],
    out_path: Path,
) -> None:
    full_img: Image.Image = result["image"]
    detections: List[DetectedPlate] = result["detections"]

    vis = full_img.copy()
    draw = ImageDraw.Draw(vis)

    for det in detections:
        draw.rectangle([det.xmin, det.ymin, det.xmax, det.ymax], outline="lime", width=4)
        draw.text(
            (det.xmin, max(0, det.ymin - 18)),
            f"P{det.plate_id} {det.confidence:.2f}",
            fill="lime",
        )

    vis.save(out_path)


example_dir = SAVED_DIR / "inference_examples"
example_dir.mkdir(parents=True, exist_ok=True)

rng = random.Random(CFG.seed)
example_samples = rng.sample(valid_valid_samples, k=min(10, len(valid_valid_samples)))

for i, sample in enumerate(example_samples):
    result = run_model1_on_image(
        image_path=sample.image_path,
        model=best_yolo_model,
        cfg=CFG,
        device=YOLO_DEVICE,
    )

    out_path = example_dir / f"example_{i:03d}_{sample.image_path.stem}.png"
    save_model1_inference_visual(result, out_path)

print("[OK] Saved example prediction visualizations to:", example_dir)

In [ ]:
input_path_str = input("Enter path to a full vehicle/scenery image: ").strip()

try:
    input_path = Path(input_path_str).expanduser().resolve()

    if not MODEL1_BEST_PT.exists():
        raise FileNotFoundError(f"Trained Model 1 best.pt not found: {MODEL1_BEST_PT}")

    inference_yolo_model = YOLO(str(MODEL1_BEST_PT))

    inference_result = run_model1_on_image(
        image_path=input_path,
        model=inference_yolo_model,
        cfg=CFG,
        device=YOLO_DEVICE,
    )

    print_boxed("Model 1 Result")
    print("Input:", input_path)
    print("Accepted:", inference_result["accepted"])

    if not inference_result["accepted"]:
        print("Rejection reason:", inference_result["rejection_reason"])

    visualize_model1_inference(
        inference_result,
        title=f"Model 1 Inference: {input_path.name}",
    )

    print_boxed("Plate Detection Metadata")
    print_plate_detection_metadata(inference_result["detections"])

    if inference_result["accepted"]:
        print_boxed("Rectification Metadata")
        display(pd.DataFrame(inference_result["rectification_info"]))

        save_crops = input("Save detected plate crops? [y/N]: ").strip().lower() == "y"

        if save_crops:
            out_dir = SAVED_DIR / "manual_inference_plate_crops" / input_path.stem
            out_dir.mkdir(parents=True, exist_ok=True)

            for i, crop in enumerate(inference_result["crops"], start=1):
                crop.save(out_dir / f"P{i}_raw.png")

            for i, crop in enumerate(inference_result["rectified_crops"], start=1):
                crop.save(out_dir / f"P{i}_rectified.png")

            print("Saved crops to:", out_dir)

except Exception as exc:
    print("[ERROR] Model 1 inference failed.")
    print(type(exc).__name__ + ":", exc)